# LangGraph Pattern for RAG Workflows
Companion notebook for the **LangGraph Pattern for RAG Workflows** section of Chapter 10.

Keep **retrieval** separate from **answer generation**, as small single-job nodes -- `rewrite_query -> retrieve -> rerank -> assess_results -> generate_answer` -- and let the graph decide what to do when search quality is weak. This treats retrieval as something to **evaluate**, not something to trust by default.

This is the book's `RAGState`, unchanged:

In [1]:
from typing import TypedDict, List, Optional


class RAGState(TypedDict):
    question: str
    rewritten_query: Optional[str]
    results: List[dict]
    reranked: List[dict]
    confidence: float
    answer: Optional[str]
    error: Optional[str]
    retry_count: int

A tiny local document set and retriever so every node is actually runnable (the MCP-backed version of `retrieve` is notebook 07 / notebook 09):

In [2]:
_DOCS = [
    {"doc_id": "policy_2025_refunds", "text": "Refunds are allowed within 30 days of purchase for annual plans."},
    {"doc_id": "billing_faq", "text": "Billing questions are answered by the support team within one business day."},
]

STOPWORDS = {"what", "is", "the", "a", "an", "for", "of", "to", "in", "on", "are", "by"}

**`rewrite_query`**: clarify a vague/short question into a better search query:

In [3]:
def rewrite_query(state: RAGState) -> dict:
    q = state["question"].strip()
    if len(q.split()) <= 3:
        q = f"{q} refund policy details"  # broaden a too-short query
    return {"rewritten_query": q}

**`retrieve`**: search using the rewritten query (stopwords filtered so common words like "the"/"for" don't create false matches):

In [4]:
def retrieve(state: RAGState) -> dict:
    query = state["rewritten_query"] or state["question"]
    q_words = [w for w in query.lower().strip("?.! ").split() if w not in STOPWORDS]
    results = []
    for doc in _DOCS:
        text_words = [w for w in doc["text"].lower().split() if w not in STOPWORDS]
        overlap = sum(1 for qw in q_words if any(tw.startswith(qw) or qw.startswith(tw) for tw in text_words))
        if overlap > 0:
            results.append({**doc, "score": overlap})
    return {"results": results}

**`rerank`** and **`assess_results`**: order candidates, then turn the top score into a 0-1 confidence:

In [5]:
def rerank(state: RAGState) -> dict:
    return {"reranked": sorted(state["results"], key=lambda r: r["score"], reverse=True)}


def assess_results(state: RAGState) -> dict:
    if not state["reranked"]:
        return {"confidence": 0.0}
    return {"confidence": min(1.0, state["reranked"][0]["score"] / 2)}

This is the book's `route_after_assess`, unchanged:

In [6]:
def route_after_assess(state):
    if not state["results"]:
        return "no_hit"
    if state["confidence"] < 0.65:
        return "low_confidence"
    return "generate"

**`generate_answer`**: a deterministic template answer built only from the top retrieved chunk (no LLM call needed to demonstrate the orchestration). **`fallback_response`** and **`broaden_and_retry`**: the two branches the book calls for when retrieval can't be trusted -- with a capped retry counter so the graph can't loop forever:

In [7]:
def generate_answer(state: RAGState) -> dict:
    top = state["reranked"][0]
    return {"answer": f"Based on {top['doc_id']}: {top['text']}"}


def fallback_response(state: RAGState) -> dict:
    return {"answer": None, "error": "no relevant documents found"}


def broaden_and_retry(state: RAGState) -> dict:
    return {
        "rewritten_query": f"{state['rewritten_query'] or state['question']} policy",
        "retry_count": state["retry_count"] + 1,
    }


def route_after_broaden(state: RAGState) -> str:
    return "give_up" if state["retry_count"] >= 2 else "retry"

## Wiring the graph
The happy path is a straight line; `no_hit` and `low_confidence` both route into a capped retry loop instead of generating from evidence the graph doesn't trust:

In [8]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(RAGState)
builder.add_node("rewrite_query", rewrite_query)
builder.add_node("retrieve", retrieve)
builder.add_node("rerank", rerank)
builder.add_node("assess_results", assess_results)
builder.add_node("generate_answer", generate_answer)
builder.add_node("fallback_response", fallback_response)
builder.add_node("broaden_and_retry", broaden_and_retry)

builder.add_edge(START, "rewrite_query")
builder.add_edge("rewrite_query", "retrieve")
builder.add_edge("retrieve", "rerank")
builder.add_edge("rerank", "assess_results")
builder.add_conditional_edges(
    "assess_results", route_after_assess,
    {"no_hit": "broaden_and_retry", "low_confidence": "broaden_and_retry", "generate": "generate_answer"},
)
builder.add_conditional_edges(
    "broaden_and_retry", route_after_broaden,
    {"retry": "retrieve", "give_up": "fallback_response"},
)
builder.add_edge("generate_answer", END)
builder.add_edge("fallback_response", END)

app = builder.compile()

In [9]:
good = app.invoke({
    "question": "What is the refund window for annual plans?",
    "results": [], "reranked": [], "retry_count": 0,
})
print("confident hit  -> retries:", good["retry_count"], " answer:", good["answer"])

weak = app.invoke({
    "question": "What is the weather forecast?",
    "results": [], "reranked": [], "retry_count": 0,
})
print("no relevant doc -> retries:", weak["retry_count"], " error:", weak["error"])

confident hit  -> retries: 0  answer: Based on policy_2025_refunds: Refunds are allowed within 30 days of purchase for annual plans.
no relevant doc -> retries: 2  error: no relevant documents found


The graph never generates from the weather-forecast query -- it retries twice (capped), then gives a clear "no relevant documents" error instead of letting the model guess. Because each step is its own node, you can inspect `results`, `reranked`, and `confidence` after the run to see exactly where a bad answer would have come from.